# Random Forest para regresión

## Tema 10 — Un experto por algoritmo

**Objetivo:** predecir la tarifa (`fare`) de un viaje en taxi de Nueva York usando un bosque aleatorio de árboles de decisión.

Este notebook sigue la estructura solicitada en la Sesión 13: explicación teórica, ejemplos cotidianos, implementación aplicada, evaluación, ventajas y desventajas, comparación con un árbol individual y preguntas de discusión.

## 1. ¿Qué problema resolvemos?

Tenemos observaciones de viajes de taxi y queremos estimar una variable numérica continua: la tarifa (`fare`). Por tanto, es un problema de **aprendizaje supervisado de regresión**.

Un modelo puede ayudar a: calcular tarifas esperadas, detectar cobros atípicos, estimar ingresos y apoyar decisiones operativas. La pregunta de negocio es: **¿cuánto debería costar aproximadamente un viaje con estas características?**

## 2. Fundamento teórico

### Árbol de decisión
Un árbol divide los datos mediante reglas como `distance <= 2.5`. En regresión, la predicción de una hoja suele ser el promedio de las observaciones que llegan a ella. Una división busca reducir la variabilidad dentro de los grupos.

Para un nodo con observaciones $S$, una medida común de impureza es el error cuadrático medio: 

$$MSE(S)=\frac{1}{|S|}\sum_{i\in S}(y_i-\bar{y}_S)^2$$

El árbol selecciona divisiones que disminuyen ese error. Un árbol demasiado profundo puede memorizar el conjunto de entrenamiento y sobreajustar.

### Bosque aleatorio
Random Forest combina muchos árboles entrenados con dos fuentes de aleatoriedad:

1. Cada árbol recibe una muestra bootstrap de las filas.
2. En cada división solo se considera un subconjunto aleatorio de variables.

Para regresión, el bosque promedia las predicciones de los $B$ árboles:

$$\hat{f}_{RF}(x)=\frac{1}{B}\sum_{b=1}^{B}T_b(x)$$

El promedio reduce la varianza: los errores particulares de árboles distintos tienden a compensarse. El modelo captura relaciones no lineales y no requiere estandarizar las variables numéricas.

### Hiperparámetros principales

| Hiperparámetro | Función | Riesgo si es demasiado alto o bajo |
|---|---|---|
| `n_estimators` | Número de árboles | Más árboles aumentan el costo, pero suelen estabilizar el resultado |
| `max_depth` | Profundidad máxima | Profundidad alta puede sobreajustar; baja puede subajustar |
| `min_samples_split` | Mínimo de observaciones para dividir | Valores altos generan árboles más simples |
| `min_samples_leaf` | Mínimo de observaciones por hoja | Ayuda a suavizar predicciones |
| `max_features` | Variables candidatas por división | Controla la diversidad entre árboles |
| `random_state` | Semilla reproducible | Permite repetir el experimento |

## 3. Ejemplos cotidianos de uso

- **Vivienda:** predecir el precio de una casa a partir del área, habitaciones y ubicación.
- **Salud:** estimar el tiempo de recuperación usando características del paciente.
- **Negocios:** pronosticar ventas o demanda de un producto.
- **Transporte:** estimar la tarifa de un taxi a partir de la distancia, pasajeros y zonas.

Elegiríamos Random Forest cuando esperamos relaciones no lineales, tenemos variables de escalas distintas y preferimos un modelo robusto con poca preparación manual.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor, plot_tree

warnings.filterwarnings('ignore')
RANDOM_STATE = 13
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', 30)

## 4. Carga y exploración de los datos

Usamos el conjunto `taxis` de Seaborn. En la primera ejecución se descarga desde el repositorio de Seaborn y se guarda localmente en `data/taxis.csv`; las siguientes ejecuciones utilizan la copia local. No usamos `total`, `tip` ni `tolls` como predictores porque pueden revelar directamente componentes de la tarifa y producir fuga de información.

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'random-forest-regression':
    DATA_PATH = PROJECT_DIR / 'data' / 'taxis.csv'
else:
    DATA_PATH = PROJECT_DIR / 'random-forest-regression' / 'data' / 'taxis.csv'

if DATA_PATH.exists():
    taxis = pd.read_csv(DATA_PATH)
else:
    taxis = sns.load_dataset('taxis')
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    taxis.to_csv(DATA_PATH, index=False)

print(f'Dimensiones: {taxis.shape[0]:,} filas y {taxis.shape[1]} columnas')
display(taxis.head())
display(taxis.dtypes.to_frame('tipo'))
display(taxis.describe(include='all').T)

In [ ]:
# Revisamos la variable objetivo y los valores faltantes.
print('Valores faltantes:')
display(taxis.isna().sum().sort_values(ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=taxis, x='fare', bins=40, kde=True, ax=axes[0])
axes[0].set_title('Distribución de la tarifa')
sns.scatterplot(data=taxis.sample(min(3000, len(taxis)), random_state=RANDOM_STATE),
                x='distance', y='fare', alpha=0.35, ax=axes[1])
axes[1].set_title('Tarifa frente a distancia')
plt.tight_layout()
plt.show()

## 5. Preparación, entrenamiento y evaluación

Conservamos variables numéricas y categóricas disponibles en el viaje. Las categóricas se transforman con one-hot encoding dentro de un `Pipeline`, evitando aplicar transformaciones usando información del conjunto de prueba.

In [ ]:
TARGET = 'fare'
LEAKAGE_COLUMNS = ['fare', 'tip', 'tolls', 'total']
FEATURES = [c for c in taxis.columns if c not in LEAKAGE_COLUMNS]
model_data = taxis[FEATURES + [TARGET]].dropna().copy()

X = model_data[FEATURES]
y = model_data[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=250, max_depth=None, min_samples_leaf=2,
        n_jobs=-1, random_state=RANDOM_STATE
    ))
])
rf_pipeline.fit(X_train, y_train)
pred_rf = rf_pipeline.predict(X_test)

def regression_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred),
    }

metrics_rf = regression_metrics(y_test, pred_rf)
pd.DataFrame([metrics_rf], index=['Random Forest']).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(x=y_test, y=pred_rf, alpha=0.35, ax=axes[0])
limits = [min(y_test.min(), pred_rf.min()), max(y_test.max(), pred_rf.max())]
axes[0].plot(limits, limits, '--', color='black')
axes[0].set(xlabel='Tarifa real', ylabel='Tarifa predicha', title='Real frente a predicho')

residuals = y_test - pred_rf
sns.histplot(residuals, bins=40, kde=True, ax=axes[1])
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Distribución de residuos')
plt.tight_layout()
plt.show()

## 6. ¿El bosque mejora frente a un solo árbol?

Un árbol individual puede alcanzar gran precisión en entrenamiento, pero suele tener alta varianza: pequeños cambios en los datos pueden cambiar mucho su estructura. El bosque busca estabilizar esa predicción mediante el promedio de muchos árboles.

In [ ]:
tree_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=12, min_samples_leaf=2, random_state=RANDOM_STATE))
])
tree_pipeline.fit(X_train, y_train)
pred_tree = tree_pipeline.predict(X_test)

comparison = pd.DataFrame([
    regression_metrics(y_test, pred_tree),
    regression_metrics(y_test, pred_rf),
], index=['Árbol individual', 'Random Forest']).round(3)
display(comparison)
comparison[['RMSE', 'MAE']].plot(kind='bar', figsize=(9, 4), rot=0, title='Error: menor es mejor')
plt.ylabel('Valor de la métrica')
plt.tight_layout()
plt.show()

## 7. Importancia de variables

La importancia por impureza indica cuánto contribuye cada variable a las divisiones del bosque. Es útil para interpretar el modelo, pero no demuestra causalidad. Para modelos finales conviene complementarla con permutation importance o SHAP.

In [ ]:
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipeline.named_steps['model'].feature_importances_
importance = (pd.DataFrame({'variable': feature_names, 'importance': importances})
              .sort_values('importance', ascending=False).head(15))
display(importance)

sns.barplot(data=importance, y='variable', x='importance')
plt.title('Variables más importantes del Random Forest')
plt.tight_layout()
plt.show()

## 8. Ajuste básico de hiperparámetros

No existe una combinación universalmente óptima. Probamos algunas alternativas y seleccionamos la que minimiza el RMSE de validación cruzada. El conjunto de prueba se mantiene separado hasta el final.

In [ ]:
param_distributions = {
    'model__n_estimators': [150, 250, 400],
    'model__max_depth': [None, 12, 20],
    'model__min_samples_leaf': [1, 2, 5],
    'model__max_features': [1.0, 'sqrt', 0.7],
}
search = RandomizedSearchCV(
    rf_pipeline, param_distributions=param_distributions, n_iter=8,
    scoring='neg_root_mean_squared_error', cv=3,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
pred_best = best_model.predict(X_test)
print('Mejores parámetros:', search.best_params_)
display(pd.DataFrame([regression_metrics(y_test, pred_best)], index=['Random Forest ajustado']).round(3))

## 9. Comparación con otros ensambles

- **Random Forest:** entrena árboles en paralelo, es robusto y suele requerir poco ajuste.
- **Gradient Boosting:** entrena árboles secuencialmente, corrigiendo errores anteriores; puede alcanzar gran precisión, pero es más sensible a hiperparámetros.
- **XGBoost:** optimiza el boosting con regularización y mejoras de velocidad; suele ser muy competitivo, aunque agrega complejidad.

Random Forest es una buena línea base cuando se busca un modelo sólido, razonablemente interpretable y fácil de entrenar.

## 10. Ventajas y desventajas

### Ventajas
- Captura relaciones no lineales e interacciones entre variables.
- Reduce la varianza frente a un árbol individual.
- No exige estandarizar variables numéricas.
- Funciona bien como modelo base y tolera diferentes escalas.
- Permite estimar importancia de variables y error OOB.

### Desventajas
- Consume más memoria y tiempo que un solo árbol.
- Es menos interpretable que un árbol pequeño o una regresión lineal.
- Puede ser menos eficiente que boosting en problemas tabulares muy ajustados.
- La importancia tradicional puede favorecer variables con muchas categorías.
- No extrapola bien fuera del rango observado de la variable objetivo.

## 11. Conclusiones

Random Forest transforma muchos árboles de alta variabilidad en una predicción más estable mediante aleatoriedad y promediado. En este caso, el desempeño debe juzgarse principalmente con RMSE, MAE y R² sobre datos no vistos, siempre vigilando la fuga de información.

La decisión final no depende únicamente de obtener el menor error: también importan el costo computacional, la explicabilidad, la facilidad de mantenimiento y el objetivo del negocio.

## 12. Preguntas de discusión

1. ¿Más árboles siempre mejoran el modelo o llega un punto en que solo aumentan el tiempo de entrenamiento?
2. ¿Qué equilibrio existe entre `max_depth`, sesgo y varianza?
3. Si `distance` aparece como la variable más importante, ¿podemos concluir que causa una tarifa mayor?
4. ¿En qué escenario elegiríamos Gradient Boosting o XGBoost en lugar de Random Forest?

## 13. Extensiones para una evaluación más completa

Además de comparar contra un árbol, usamos una línea base sencilla: predecir siempre la tarifa promedio del entrenamiento. Un modelo es útil si mejora esta referencia. También activamos la evaluación **out-of-bag (OOB)**, que aprovecha las observaciones que cada árbol no recibió en su muestra bootstrap.

In [ ]:
baseline_pred = np.repeat(y_train.mean(), len(y_test))
baseline_metrics = regression_metrics(y_test, baseline_pred)

rf_oob_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=250, min_samples_leaf=2, max_depth=None,
        bootstrap=True, oob_score=True, n_jobs=-1, random_state=RANDOM_STATE
    ))
])
rf_oob_pipeline.fit(X_train, y_train)
pred_oob_model = rf_oob_pipeline.predict(X_test)
oob_r2 = rf_oob_pipeline.named_steps['model'].oob_score_

final_comparison = pd.DataFrame([
    baseline_metrics,
    regression_metrics(y_test, pred_tree),
    regression_metrics(y_test, pred_oob_model),
    regression_metrics(y_test, pred_best),
], index=['Línea base: promedio', 'Árbol individual', 'Random Forest + OOB', 'Random Forest ajustado']).round(3)
display(final_comparison)
print(f'R² OOB del bosque: {oob_r2:.3f}')

## 14. Análisis de errores

Una métrica agregada no muestra si el modelo falla más en viajes cortos o largos. Revisamos el error absoluto por grupos de distancia y observamos algunos casos con mayor error.

In [ ]:
error_analysis = X_test.copy()
error_analysis['fare_real'] = y_test
error_analysis['fare_predicha'] = pred_best
error_analysis['error_absoluto'] = (error_analysis['fare_real'] - error_analysis['fare_predicha']).abs()
error_analysis['grupo_distancia'] = pd.cut(
    error_analysis['distance'], bins=[-np.inf, 1, 3, 7, np.inf],
    labels=['corta', 'media-corta', 'media-larga', 'larga']
)
display(error_analysis.groupby('grupo_distancia', observed=True)['error_absoluto'].agg(['count', 'mean', 'median']).round(3))

worst_cases = error_analysis.sort_values('error_absoluto', ascending=False).head(10)
display(worst_cases[['distance', 'passengers', 'fare_real', 'fare_predicha', 'error_absoluto']])

sns.boxplot(data=error_analysis, x='grupo_distancia', y='error_absoluto')
plt.title('Error absoluto por grupo de distancia')
plt.xlabel('Grupo de distancia')
plt.ylabel('Error absoluto')
plt.tight_layout()
plt.show()

## 15. Cierre frente a los requisitos de la Sesión 13

| Requisito | Evidencia en este notebook |
|---|---|
| Explicación teórica completa | Árboles, ensamble, ecuación, aleatoriedad e hiperparámetros |
| Ejemplos cotidianos | Vivienda, salud, negocios y transporte |
| Ejemplo práctico | Datos `taxis`, exploración, preparación, entrenamiento y predicción |
| Métrica apropiada | MAE, RMSE, R² y comparación con línea base |
| Visualización | Distribución, real vs. predicho, residuos, importancia y errores por distancia |
| Ventajas y desventajas | Sección dedicada con comparación frente a boosting |
| Discusión | Cuatro preguntas sobre hiperparámetros, interpretación y elección del modelo |

**Conclusión para la presentación:** Random Forest es una alternativa sólida para relaciones no lineales y datos tabulares. Su principal beneficio frente a un árbol individual es la reducción de varianza mediante el promedio de árboles; su principal costo es una menor interpretabilidad y mayor consumo computacional. La selección final debe basarse en el desempeño sobre datos no vistos y en las necesidades del problema.